# PFEM/Transolver Training -- B1 x {NH, MR, AB}, B2 x {NH, MR, AB}
**Colab GPU runtime required (Runtime > Change runtime type > GPU)**

Runs the full PFEM pipeline (Wang et al., "Pretrain finite element method",
JMPS 214 (2026) 106682) for all 6 benchmark cases: B1 (unit square,
top-edge traction, fixed bottom) and B2 (quarter ring, R_in=1/R_out=2,
internal pressure, symmetry BCs), each with three hyperelastic energy
densities (Neo-Hookean, Mooney-Rivlin, Arruda-Boyce).

This notebook runs in two phases, per the advisor's guidance that blindly
training all 6 cases for 10000 epochs each is neither necessary nor
efficient (validation was too sparse to tell, and the best result so far
came around epoch 1000, with later epochs possibly hurt by optimization
instability or the LR schedule rather than genuine underfitting):

- **Phase 1 (screening)**: on one representative case (B1 x Neo-Hookean),
  screen a few real mini-batch sizes (a true parallel batch over samples
  sharing the same mesh, not gradient accumulation) for a short, fixed
  epoch budget, validating every 25-50 epochs and tracking wall-clock
  time, GPU peak memory, and optimizer steps alongside validation error.
  The most promising batch size is then continued to a larger epoch
  budget with early stopping (on validation error, not a fixed epoch
  count) -- producing a small comparison table and one concrete,
  automatically-chosen training protocol.
- **Phase 2 (the other 5 cases)**: that same protocol (batch size,
  validation cadence, early-stopping patience) is applied uniformly to
  the remaining cases -- no manual per-case tuning, per the advisor's
  explicit goal of "a robust and efficient procedure," not one-off tuning.

For each case, the pipeline is:
1. Generate a FEM ground-truth dataset (Total-Lagrangian Newton-Raphson,
   Q4 elements) via `omar_pfem/data/data_generate_B{1,2}.py`, parallelized
   across CPU cores -- skipped if the dataset already exists on disk.
2. Convert it to the Transolver NPZ format via `convert_B{1,2}_quad.py`.
3. Train a physics-informed Transolver (`omar_pfem/train_B{1,2}.py`) by
   minimizing total potential energy (Pi = U - W, no labeled-data loss),
   with n_hidden=256, n_layers=4, n_heads=8, slice_num=128 (PFEM's own
   architecture defaults) -- batch size and epoch budget now come from
   Phase 1 instead of being fixed in advance.

**Scale note**: PFEM's own reference script defaults to ntrain=800,
ntest=200 -- generating that many samples per case is still a real cost
independent of the training-protocol question above:
- **Data generation**: measured on a 4-core CPU at the reference 21x21 Q4
  mesh, generating each sample (a 10-step Newton-Raphson solve) took
  ~8s/sample with 4 parallel workers -- so a 1000-sample dataset
  (NTRAIN+NTEST) is roughly 2-2.5 CPU-hours per case, ~14 hours for all 6.
  Colab's own CPU allocation may differ from this benchmark.

Given that, this will likely span multiple Colab sessions. That's expected
and handled: every case checkpoints its model periodically and **resumes
from its own latest checkpoint** if this notebook is re-run (whether
because the runtime disconnected mid-case, or because you're continuing
across multiple sessions) -- so re-running `Runtime > Run all` after a
disconnect always makes forward progress instead of starting over.
Finished cases (`model_final.pt` or an `EARLY_STOPPED` marker on disk) are
skipped entirely, finished datasets (an NPZ already on disk) are never
regenerated, and a completed Phase 1 (a saved `training_protocol.json`)
is never re-screened.

If you'd rather trade fidelity for a faster full pass, lower
`TARGET_SAMPLES` in the config cell below -- everything else adapts
automatically.


## Cell 1 - Install dependencies

Colab's preinstalled `torch` already has CUDA support -- it is deliberately
NOT reinstalled here (a bare `pip install torch` risks silently replacing
it with a CPU-only wheel). Only the packages PFEM's Transolver model and
this pipeline actually need on top of Colab's base image are installed:
`einops`/`timm` (Transolver architecture), `h5py` (FEM dataset storage),
`jax` (autodiff-derived PK1 stress/tangent for Mooney-Rivlin/Arruda-Boyce
in the FEM generator -- CPU-only use, small dense tensors, no GPU needed).

Uses `{sys.executable} -m pip` rather than bare `!pip` -- on some Colab
runtimes the shell's `pip` has been observed to resolve to a different
Python than the notebook kernel itself, which makes packages "install
successfully" yet still fail to import.


In [ ]:
import sys
!{sys.executable} -m pip install -q einops timm h5py jax tqdm
print('Done - continue to Cell 2')


## Cell 2 - Clone the repo

If the repo is private, paste a GitHub personal access token as the value
of `GITHUB_TOKEN` below. Leave it as `""` if the repo is public -- never
type a literal `<TOKEN>` placeholder into the URL (`<`/`>` are bash
redirection operators and will break the clone before git even runs).

Always does a clean re-clone of the CODE (removes any previous
`/content/OMAR` first) so a broken partial clone can't linger -- but
generated datasets and results live elsewhere (local disk / Google Drive,
see Cells 3-4), **outside** the cloned repo, so re-cloning never discards
training progress.


In [ ]:
import os
import shutil
import sys

os.chdir('/content')

GITHUB_TOKEN = ""  # <-- paste your token between the quotes if the repo is private; leave "" if public
BRANCH = "claude/claude-code-question-d307wp"
REPO_URL = (f"https://{GITHUB_TOKEN}@github.com/suhibamro/omar.git" if GITHUB_TOKEN
            else "https://github.com/suhibamro/omar.git")

if os.path.exists('/content/OMAR'):
    shutil.rmtree('/content/OMAR')

!git clone -b {BRANCH} {REPO_URL} /content/OMAR

WORK_DIR = '/content/OMAR/Practical_Examples'
if not os.path.isdir(WORK_DIR):
    raise SystemExit(
        'ERROR: clone failed -- /content/OMAR/Practical_Examples does not exist.\n'
        'Scroll up to the "git clone" output above for the actual error.\n'
        'Common cause: the repo is private and GITHUB_TOKEN is still "".'
    )

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

pfem_ok = os.path.isdir(os.path.join(WORK_DIR, 'omar_pfem'))
if pfem_ok:
    print('Clone OK: omar_pfem/ found under Practical_Examples/.')
else:
    raise SystemExit('ERROR: omar_pfem/ not found -- clone looks incomplete or wrong branch.')

import torch
print(f'torch {torch.__version__} | cuda available={torch.cuda.is_available()}', end=' ')
if torch.cuda.is_available():
    print(f'| device={torch.cuda.get_device_name(0)}')
else:
    print()
    print('WARNING: no GPU detected -- go to Runtime > Change runtime type and select a GPU, '
          'then Runtime > Restart session and re-run from Cell 1.')


## Cell 3 - Mount Google Drive (persist training progress across full disconnects)

Colab's local `/content` disk only survives a *reconnect* to the same
runtime -- it does NOT survive a full runtime reset/reassignment (hitting
the session time limit, "Factory reset runtime", or Colab reclaiming an
idle VM). Given this pipeline can realistically run for many hours to
multiple days across several sessions, that's a real risk for the
expensive part of the run: GPU training checkpoints.

So results (`RESULTS_DIR` below) are written to Google Drive, which
survives any of the above. FEM datasets (`DATA_DIR`) stay on local
`/content` disk instead -- deliberately NOT on Drive -- because Drive is
FUSE-mounted (each file write is a network round-trip), and the data
generator does hundreds of small sequential writes into one HDF5 file per
case; on Drive that overhead would meaningfully slow down the part of the
pipeline that's cheap to just regenerate if lost (a bounded few CPU-hours,
deterministic given the same seed), whereas losing GPU training progress
is not cheap to redo. If prompted, click through the Google auth flow --
this notebook will not work with your data without that authorization.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted -- continue to Cell 4')


## Cell 4 - Configuration

Edit these to trade off dataset/training scale against wall-clock time.
`TARGET_SAMPLES` splits into `NTRAIN`/`NTEST` the same way as PFEM's own
reference script (800/200 by default -- lower this if a full 1000-sample
FEM generation pass is too slow on your Colab CPU allocation).

The screening-study settings (`SCREEN_*`) control Phase 1 only; its
outcome (`training_protocol.json`) then drives every case in Phase 2, so
there's nothing to configure per-case beyond dataset scale.


In [ ]:
import os

DATA_DIR = '/content/pfem_data'                                  # local disk: fast, regenerable if lost
RESULTS_DIR = '/content/drive/MyDrive/pfem_run/results'          # Google Drive: durable across full resets
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---- Dataset scale (PFEM's own reference: ntrain=800, ntest=200) ----
NTRAIN = 800
NTEST = 200
TARGET_SAMPLES = NTRAIN + NTEST   # total FEM samples generated per case

# ---- Mesh resolution (PFEM's own reference default for the beam case) ----
MESH_N = 21   # Nx=Ny=21 for B1; Ntheta=Nr=21 for B2

N_WORKERS = os.cpu_count()

# ---- Phase 1 (screening study on B1 x Neo-Hookean) ----
SCREEN_BATCH_SIZES = "4,8,16"   # comma string, passed straight to screening_study.py
SCREEN_EPOCHS = 400             # short, fixed budget per batch size (advisor: ~250-500)
VALIDATE_EVERY = 25             # advisor: validate every 25-50 epochs
CONTINUE_EPOCHS = 2000          # upper bound for the winning config (advisor: ~1000-2000)
EARLY_STOP_PATIENCE = 8         # validation events with no improvement before stopping
EARLY_STOP_MIN_DELTA = 1e-4

GEOMETRIES = ["B1", "B2"]
MATERIALS = ["neo_hookean", "mooney_rivlin", "arruda_boyce"]
CASES = [(g, m) for g in GEOMETRIES for m in MATERIALS]
SCREENING_CASE = ("B1", "neo_hookean")

print(f'CPU workers for data generation: {N_WORKERS}')
print(f'Target samples/case: {TARGET_SAMPLES} (ntrain={NTRAIN}, ntest={NTEST})')
print(f'Screening case: {SCREENING_CASE}, batch_sizes={SCREEN_BATCH_SIZES}, '
      f'screen_epochs={SCREEN_EPOCHS}, validate_every={VALIDATE_EVERY}')
print(f'Cases: {CASES}')


## Cell 5 - Helpers: streaming subprocess runner + shared dataset step

`run_streaming` runs a command as a subprocess and prints its stdout live
(unbuffered, `python -u`) instead of buffering until the process exits --
essential for a run that may take hours, so progress is visible the whole
time instead of appearing in one dump at the end (or not at all, if the
cell is interrupted). `ensure_dataset` is shared between the screening
study (Cell 6) and the main per-case loop (Cell 8) so the FEM dataset for
B1 x Neo-Hookean is only ever generated once.


In [ ]:
import subprocess
import time
import json as _json

def run_streaming(cmd, cwd=None):
    print(f"$ {' '.join(cmd)}")
    proc = subprocess.Popen(
        cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {' '.join(cmd)}")


def case_name(geometry, material):
    return f"{geometry}_{material}"


def ensure_dataset(geometry, material):
    """Generates + converts the FEM dataset for one case if it isn't
    already on disk. Returns the NPZ path."""
    name = case_name(geometry, material)
    h5_dir = os.path.join(DATA_DIR, f"fem_{name}")
    npz_dir = os.path.join(DATA_DIR, f"npz_{name}")
    npz_path = os.path.join(npz_dir, "hyperelastic_training_data_q4.npz")

    if os.path.exists(npz_path):
        print(f"[{name}] NPZ dataset already exists at {npz_path}, skipping generation.")
        return npz_path

    t0 = time.time()
    if geometry == "B1":
        gen_module = "omar_pfem.data.data_generate_B1"
        gen_args = ["--Nx", str(MESH_N), "--Ny", str(MESH_N)]
        conv_module = "omar_pfem.data.convert_B1_quad"
    else:
        gen_module = "omar_pfem.data.data_generate_B2"
        gen_args = ["--Ntheta", str(MESH_N), "--Nr", str(MESH_N)]
        conv_module = "omar_pfem.data.convert_B2_quad"

    run_streaming([
        sys.executable, "-u", "-m", gen_module,
        "--num_index", "1", "--num_samples", str(TARGET_SAMPLES),
        *gen_args, "--material", material,
        "--n_workers", str(N_WORKERS),
        "--out_dir", h5_dir,
    ], cwd=WORK_DIR)

    run_streaming([
        sys.executable, "-u", "-m", conv_module,
        "--h5_dir", h5_dir, "--out_dir", npz_dir,
    ], cwd=WORK_DIR)

    print(f"[{name}] Dataset ready after {time.time()-t0:.0f}s")
    return npz_path


## Cell 6 - Phase 1: screening study (B1 x Neo-Hookean)

Runs `omar_pfem/screening_study.py`, which:
1. Trains B1 x Neo-Hookean at each of `SCREEN_BATCH_SIZES` for a short,
   fixed `SCREEN_EPOCHS` budget, validating every `VALIDATE_EVERY` epochs
   and logging validation error, wall-clock time, GPU peak memory, and
   optimizer-step count for each.
2. Picks the batch size with the lowest best validation error and
   continues *that same run* (resumed from its own screening checkpoint)
   to `CONTINUE_EPOCHS`, this time with early stopping
   (`EARLY_STOP_PATIENCE` validation events with no improvement).

`--final_out_dir` points the continuation directly at this case's real
results directory, so it doubles as B1 x Neo-Hookean's actual training run
-- Cell 8's main loop later finds it already done and skips it, no
duplicated work.

If a previous run already finished Phase 1 (`training_protocol.json`
exists), this cell loads that instead of re-screening -- so a Colab
disconnect never repeats the screening study itself.


In [ ]:
protocol_path = os.path.join(RESULTS_DIR, "training_protocol.json")

if os.path.exists(protocol_path):
    with open(protocol_path) as f:
        PROTOCOL = _json.load(f)
    print(f"Phase 1 already completed -- loaded protocol from {protocol_path}:")
    print(_json.dumps(PROTOCOL, indent=2))
else:
    screen_geometry, screen_material = SCREENING_CASE
    screen_npz_path = ensure_dataset(screen_geometry, screen_material)
    screen_out_dir = os.path.join(DATA_DIR, "screening_" + case_name(*SCREENING_CASE))
    final_out_dir = os.path.join(RESULTS_DIR, case_name(*SCREENING_CASE))

    run_streaming([
        sys.executable, "-u", "-m", "omar_pfem.screening_study",
        "--path", screen_npz_path,
        "--geometry", screen_geometry, "--material", screen_material,
        "--ntrain", str(NTRAIN), "--ntest", str(NTEST),
        "--batch_sizes", SCREEN_BATCH_SIZES,
        "--screen_epochs", str(SCREEN_EPOCHS),
        "--validate_every", str(VALIDATE_EVERY),
        "--continue_epochs", str(CONTINUE_EPOCHS),
        "--continue_early_stop_patience", str(EARLY_STOP_PATIENCE),
        "--continue_early_stop_min_delta", str(EARLY_STOP_MIN_DELTA),
        "--final_out_dir", final_out_dir,
        "--out_dir", screen_out_dir,
    ], cwd=WORK_DIR)

    with open(os.path.join(screen_out_dir, "screening_summary.json")) as f:
        screening_rows = _json.load(f)
    winner_bs = min(
        (r for r in screening_rows if r["best_val_error"] is not None),
        key=lambda r: r["best_val_error"],
    )["batch_size"]

    PROTOCOL = {
        "batch_size": winner_bs,
        "validate_every": VALIDATE_EVERY,
        "early_stop_patience": EARLY_STOP_PATIENCE,
        "early_stop_min_delta": EARLY_STOP_MIN_DELTA,
        "epochs": CONTINUE_EPOCHS,
    }
    with open(protocol_path, "w") as f:
        _json.dump(PROTOCOL, f, indent=2)

    print("\nPhase 1 done. Screening summary:")
    with open(os.path.join(screen_out_dir, "screening_summary.md")) as f:
        print(f.read())
    print(f"\nChosen protocol (saved to {protocol_path}):")
    print(_json.dumps(PROTOCOL, indent=2))


## Cell 7 - Run the remaining 5 cases with the chosen protocol

Applies `PROTOCOL` from Phase 1 uniformly -- no per-case tuning, per the
advisor's explicit goal. B1 x Neo-Hookean is skipped here since Phase 1
already trained it (its `model_final.pt`/`EARLY_STOPPED` marker is
already in place).


In [ ]:
def run_one_case(geometry, material):
    name = case_name(geometry, material)
    out_dir = os.path.join(RESULTS_DIR, name)

    print(f"\n{'='*80}\n===== CASE: {name} =====\n{'='*80}")

    if os.path.exists(os.path.join(out_dir, "model_final.pt")) or \
       os.path.exists(os.path.join(out_dir, "EARLY_STOPPED")):
        print(f"[{name}] already fully trained (model_final.pt or EARLY_STOPPED found), skipping.")
        return

    npz_path = ensure_dataset(geometry, material)

    train_module = "omar_pfem.train_B1" if geometry == "B1" else "omar_pfem.train_B2"
    t1 = time.time()
    run_streaming([
        sys.executable, "-u", "-m", train_module,
        "--path", npz_path,
        "--material", material,
        "--ntrain", str(NTRAIN), "--ntest", str(NTEST),
        "--epochs", str(PROTOCOL["epochs"]),
        "--batch_size", str(PROTOCOL["batch_size"]),
        "--validate_every", str(PROTOCOL["validate_every"]),
        "--save_every", str(PROTOCOL["validate_every"]),
        "--early_stop_patience", str(PROTOCOL["early_stop_patience"]),
        "--early_stop_min_delta", str(PROTOCOL["early_stop_min_delta"]),
        "--print_every", "999999",
        "--out_dir", out_dir,
    ], cwd=WORK_DIR)
    print(f"[{name}] Training finished after {time.time()-t1:.0f}s")


for geometry, material in CASES:
    if (geometry, material) == SCREENING_CASE:
        continue  # already trained as part of Phase 1
    run_one_case(geometry, material)

print("\nAll cases either completed or already were -- see per-case status above.")


## Cell 8 - Summary: metrics + loss curves across all 6 cases

Reads each case's `metrics_history.json` -- works even for cases that are
still mid-run or were only partially completed before a disconnect.


In [ ]:
import numpy as np
from matplotlib import pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
summary_rows = []

for ax, (geometry, material) in zip(axes.flat, CASES):
    name = case_name(geometry, material)
    metrics_path = os.path.join(RESULTS_DIR, name, "metrics_history.json")
    if not os.path.exists(metrics_path):
        ax.set_title(f"{name}\n(no data yet)")
        continue

    with open(metrics_path) as f:
        history = _json.load(f)
    if not history:
        ax.set_title(f"{name}\n(empty history)")
        continue

    epochs = [h["epoch"] for h in history]
    val_err = [h["val_error"] for h in history]

    ax.semilogy(epochs, val_err, label="val_error")
    ax.set_title(f"{name} (epoch {epochs[-1]})")
    ax.set_xlabel("epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)

    summary_rows.append((name, epochs[-1], history[-1]["mean_rel_L2_u"], history[-1]["mean_rel_L2_v"]))

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "all_cases_loss_curves.png"), dpi=150)
plt.show()

print(f"{'case':<20s} {'last epoch':>10s} {'RelL2(u)':>12s} {'RelL2(v)':>12s}")
for name, ep, u, v in summary_rows:
    print(f"{name:<20s} {ep:>10d} {u:>12.3e} {v:>12.3e}")


## Cell 9 - Zip and download all results (optional)

Results already live on Google Drive (`RESULTS_DIR`), so this is just a
convenience if you want a local zip copy too -- not required for safety.


In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive('/content/pfem_results', 'zip', RESULTS_DIR)
print(f"Archived {RESULTS_DIR} -> {archive_path}")
files.download(archive_path)
